# **1. Perkenalan Dataset**

Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.

Dataset yang digunakan pada eksperimen ini adalah **Iris Dataset**, salah satu dataset paling populer dalam dunia machine learning. Dataset ini pertama kali diperkenalkan oleh Ronald Fisher pada tahun 1936. Iris dataset berisi 150 sampel dari tiga spesies bunga iris (*Iris setosa*, *Iris versicolor*, dan *Iris virginica*) dengan empat fitur pengukuran:
- **sepal_length**: Panjang sepal (cm)
- **sepal_width**: Lebar sepal (cm)
- **petal_length**: Panjang petal (cm)
- **petal_width**: Lebar petal (cm)
- **species**: Label kelas (setosa, versicolor, virginica)

**Tujuan**: Membangun model klasifikasi untuk memprediksi spesies bunga iris berdasarkan dimensi sepal dan petal.

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimpor')

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [ ]:
df = pd.read_csv('../iris_raw.csv')

print('Shape dataset:', df.shape)
print('\n5 data pertama:')
df.head()

In [ ]:
print('5 data terakhir:')
df.tail()

In [ ]:
print('Informasi dataset:')
df.info()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
print('Statistik deskriptif:')
df.describe()

In [ ]:
print('Jumlah nilai kosong per kolom:')
print(df.isnull().sum())
print('\nTotal missing values:', df.isnull().sum().sum())

In [ ]:
print('Jumlah data duplikat:', df.duplicated().sum())

In [ ]:
print('Distribusi kelas (species):')
print(df['species'].value_counts())

plt.figure(figsize=(6, 4))
df['species'].value_counts().plot(kind='bar', color=['steelblue', 'coral', 'green'])
plt.title('Distribusi Kelas Iris')
plt.xlabel('Species')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, col in enumerate(feature_cols):
    ax = axes[i // 2][i % 2]
    for species in df['species'].unique():
        ax.hist(df[df['species'] == species][col], alpha=0.6, label=species, bins=15)
    ax.set_title(f'Distribusi {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frekuensi')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
correlation_matrix = df[feature_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix Fitur Iris')
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df, hue='species', diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Pairplot Fitur Iris per Species', y=1.02)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, col in enumerate(feature_cols):
    ax = axes[i // 2][i % 2]
    df.boxplot(column=col, by='species', ax=ax)
    ax.set_title(f'Boxplot {col} per Species')
    ax.set_xlabel('Species')
    ax.set_ylabel(col)
plt.suptitle('')
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
df_processed = df.copy()
print('Shape sebelum preprocessing:', df_processed.shape)

In [ ]:
df_processed = df_processed.dropna()
print('Shape setelah menghapus missing values:', df_processed.shape)

In [ ]:
df_processed = df_processed.drop_duplicates()
print('Shape setelah menghapus duplikat:', df_processed.shape)

In [ ]:
Q1 = df_processed[feature_cols].quantile(0.25)
Q3 = df_processed[feature_cols].quantile(0.75)
IQR = Q3 - Q1

mask = ~((df_processed[feature_cols] < (Q1 - 1.5 * IQR)) |
         (df_processed[feature_cols] > (Q3 + 1.5 * IQR))).any(axis=1)

df_processed = df_processed[mask]
print('Shape setelah menghapus outlier:', df_processed.shape)

In [ ]:
le = LabelEncoder()
df_processed['species'] = le.fit_transform(df_processed['species'])

print('Mapping label encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls} -> {i}')

In [ ]:
scaler = StandardScaler()
df_processed[feature_cols] = scaler.fit_transform(df_processed[feature_cols])

print('Statistik setelah standarisasi:')
df_processed[feature_cols].describe().round(4)

In [ ]:
print('5 data pertama setelah preprocessing:')
df_processed.head()

In [ ]:
import os
os.makedirs('iris_preprocessing', exist_ok=True)
df_processed.to_csv('iris_preprocessing/iris_preprocessing.csv', index=False)
print('Data preprocessing berhasil disimpan ke iris_preprocessing/iris_preprocessing.csv')
print('Shape akhir:', df_processed.shape)